# Miso TTS 8B on Google Colab

Run [Miso TTS 8B](https://huggingface.co/MisoLabs/MisoTTS) text-to-speech in Colab.

**Before you start**

1. **Runtime → Change runtime type → GPU**
2. Prefer **A100 (40 GB)** or **L4 (24 GB)** (Colab Pro+). A **T4 (16 GB)** will usually OOM.
3. First run downloads ~35 GB of model weights (Miso + Mimi + SilentCipher). Mount Google Drive (optional) to persist the cache across sessions.

Model card: [MisoLabs/MisoTTS](https://huggingface.co/MisoLabs/MisoTTS) · Repo: [MisoLabsAI/MisoTTS](https://github.com/MisoLabsAI/MisoTTS)

In [1]:
import os
import sys

import torch

os.environ.setdefault("NO_TORCH_COMPILE", "1")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "120")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU required. In Colab: Runtime → Change runtime type → Hardware accelerator: GPU"
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")

if vram_gb < 20:
    print(
        "\n⚠️  Warning: Miso TTS 8B typically needs ≥24 GB VRAM (L4/A100). "
        "T4 (16 GB) often runs out of memory. Switch GPU type under Runtime settings."
    )
elif vram_gb < 28:
    print("\n✓ VRAM looks sufficient for bf16 inference (L4-class).")
else:
    print("\n✓ VRAM looks comfortable (A100-class).")

GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.5 GB

✓ VRAM looks comfortable (A100-class).


## Optional: persist Hugging Face cache on Google Drive

Set `USE_DRIVE_CACHE = True` to avoid re-downloading ~35 GB on every new session.

In [ ]:
USE_DRIVE_CACHE = False  # @param {type:"boolean"}
DRIVE_CACHE_DIR = "/content/drive/MyDrive/miso-tts-hf-cache"

if USE_DRIVE_CACHE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    os.environ["HF_HOME"] = DRIVE_CACHE_DIR
    os.environ["HUGGINGFACE_HUB_CACHE"] = DRIVE_CACHE_DIR
    print(f"HF cache: {DRIVE_CACHE_DIR}")
else:
    print("Using ephemeral Colab disk for HF cache (downloads repeat each session).")

## Install dependencies

Clones the repo and installs the package (PyTorch 2.4, Moshi/Mimi, SilentCipher, etc.). Takes a few minutes.

In [3]:
REPO_URL = "https://github.com/MisoLabsAI/MisoTTS.git"
WORKDIR = "/content/MisoTTS"
BRANCH = "main"  # @param {type:"string"}

if os.path.isdir(WORKDIR):
    print(f"Repo already at {WORKDIR}")
else:
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {WORKDIR}

os.chdir(WORKDIR)
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)

print(f"Working directory: {os.getcwd()}")

Cloning into '/content/MisoTTS'...
remote: Enumerating objects: 16, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 16 (delta 0), reused 9 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (16/16), 155.15 KiB | 31.03 MiB/s, done.
Working directory: /content/MisoTTS


In [9]:
%pip install -q -e .
%pip install -q safetensors torchaudio==2.4.0 torchvision==0.19.0 huggingface-hub==0.28.1

import os
os.kill(os.getpid(), 9)

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for miso-tts (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.28.1 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.28.1 which is incompatible.


## Load model

Downloads `MisoLabs/MisoTTS` (~33 GB) on first run, then loads Mimi + SilentCipher. This cell can take 10–30+ minutes the first time.

In [2]:
import gc

import torch

from generator import DEFAULT_MISO_TTS_REPO_ID, load_miso_8b

device = "cuda"
model_repo = DEFAULT_MISO_TTS_REPO_ID

if "generator" in globals():
    del generator
    gc.collect()
    torch.cuda.empty_cache()

print(f"Loading {model_repo} on {device} (bfloat16)...")
print("First run: large download from Hugging Face Hub.")

generator = load_miso_8b(device=device, model_path_or_repo_id=model_repo)

allocated_gb = torch.cuda.max_memory_allocated() / (1024**3)
print(f"Sample rate: {generator.sample_rate} Hz")
print(f"GPU memory allocated after load: {allocated_gb:.2f} GB")

Loading MisoLabs/MisoTTS on cuda (bfloat16)...
First run: large download from Hugging Face Hub.


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.2-1B.
403 Client Error. (Request ID: Root=1-6a21fdcc-004d8b1a1b2a6e9c3dc991ab;9324ed4f-cb31-4089-9277-2d5f646cfd16)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.2-1B/resolve/main/config.json.
Your request to access model meta-llama/Llama-3.2-1B is awaiting a review from the repo authors.

## Single utterance

Edit `text` and `speaker` (integer speaker id for multi-speaker dialogs).

In [ ]:
import torchaudio
from IPython.display import Audio, display

text = "Hello from Miso TTS on Colab."
speaker = 0
max_audio_length_ms = 10_000  # @param {type:"integer"}
temperature = 0.9  # @param {type:"number"}
topk = 50  # @param {type:"integer"}

print(f"Generating: {text!r}")
audio = generator.generate(
    text=text,
    speaker=speaker,
    context=[],
    max_audio_length_ms=max_audio_length_ms,
    temperature=temperature,
    topk=topk,
)

out_path = "/content/miso_single.wav"
torchaudio.save(out_path, audio.unsqueeze(0).cpu(), generator.sample_rate)
print(f"Saved: {out_path}")
display(Audio(out_path))

## Multi-turn conversation

Each line conditions on prior generated segments (same as `run_misotts.py`).

In [ ]:
import torch
import torchaudio
from IPython.display import Audio, display

from generator import Segment

conversation = [
    {"text": "I'm just honestly not that into him, you know?", "speaker_id": 0},
    {"text": "Yeah, I get it.", "speaker_id": 1},
    {
        "text": (
            "And it's just like, I know I said I'd go out with you and stuff, "
            "but it's just like, I can't you know."
        ),
        "speaker_id": 0,
    },
    {"text": "Yeah, honestly that's totally fair.", "speaker_id": 1},
]

generated_segments = []
for utterance in conversation:
    print(f"Speaker {utterance['speaker_id']}: {utterance['text']}")
    audio_tensor = generator.generate(
        text=utterance["text"],
        speaker=utterance["speaker_id"],
        context=generated_segments,
        max_audio_length_ms=10_000,
    )
    generated_segments.append(
        Segment(
            text=utterance["text"],
            speaker=utterance["speaker_id"],
            audio=audio_tensor,
        )
    )

all_audio = torch.cat([seg.audio for seg in generated_segments], dim=0)
out_path = "/content/miso_conversation.wav"
torchaudio.save(out_path, all_audio.unsqueeze(0).cpu(), generator.sample_rate)
print(f"Saved: {out_path}")
display(Audio(out_path))

## Optional: voice prompt (cloning)

Upload a short WAV prompt and provide its transcript. Resample to the model rate automatically.

In [ ]:
from google.colab import files
from IPython.display import Audio, display

import torchaudio

from generator import Segment

RUN_PROMPTED = False  # @param {type:"boolean"}

if not RUN_PROMPTED:
    print("Set RUN_PROMPTED = True and re-run to upload a prompt WAV.")
else:
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file uploaded.")
    prompt_path = "/content/" + next(iter(uploaded))
    prompt_transcript = "This is the transcript for the prompt audio."
    text_to_synthesize = "This is the next sentence to synthesize in the same voice."
    speaker = 0

    prompt_audio, sr = torchaudio.load(prompt_path)
    prompt_audio = torchaudio.functional.resample(
        prompt_audio.mean(dim=0),
        orig_freq=sr,
        new_freq=generator.sample_rate,
    )

    context = [
        Segment(speaker=speaker, text=prompt_transcript, audio=prompt_audio),
    ]
    audio = generator.generate(
        text=text_to_synthesize,
        speaker=speaker,
        context=context,
        max_audio_length_ms=10_000,
    )

    out_path = "/content/miso_prompted.wav"
    torchaudio.save(out_path, audio.unsqueeze(0).cpu(), generator.sample_rate)
    print(f"Saved: {out_path}")
    display(Audio(out_path))

## Download WAV to your computer

In [ ]:
from google.colab import files

for path in ("/content/miso_single.wav", "/content/miso_conversation.wav", "/content/miso_prompted.wav"):
    if os.path.isfile(path):
        files.download(path)